In [17]:
import os
import json
import re
from transformers import AutoTokenizer

In [18]:
my_cache_dir = r"E:\Models"
all_chunks = []
BLOCK_SIZE = 512
STRIDE = 128
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T", cache_dir=my_cache_dir)

In [19]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
# def tokenize_fn(examples):
#     tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
#     tokens["labels"] = tokens["input_ids"].copy()
#     return tokens

In [20]:
# Using absolute paths to ensure the notebook runs correctly regardless of the Jupyter working directory
notebook_dir = os.path.abspath("")
nest_path = os.path.join(notebook_dir, "docs", "nest")
integration_path = os.path.join(nest_path, "integration")
sample_path = os.path.join(nest_path, "sample")

print("Notebook Directory:", notebook_dir)
print("Nest path exists:", os.path.exists(nest_path))
print("Integration path exists:", os.path.exists(integration_path))

Notebook Directory: E:\Docs_tunning
Nest path exists: True
Integration path exists: True


In [21]:
nest_path_integration = "./docs/nest/integration"
many_files = ["./docs/nest/integration", "./docs/nest/sample"]
file_directories = []

def add_file_directories(purpose, tasks, path):
    file_directories.append({
        "purpose": re.sub(r'[0-9]', '', purpose),
        "task": tasks,
        "path": path
    })

more = 0

for nest_type in many_files:
    print(nest_type)
    if not os.path.exists(nest_type):
        continue
        
    for item in os.listdir(nest_type):
        item_path = os.path.join(nest_type, item)
        src_path = os.path.join(item_path, "src")
        
        if os.path.isdir(item_path) and os.path.exists(src_path):
            for root, dirs, files in os.walk(src_path):
                for file in files:
                    if file.endswith(".ts"):
                        file_path = os.path.join(root, file)
                        rel_path = os.path.relpath(file_path, src_path)
                        path_parts = rel_path.split(os.sep)
                        
                        if len(path_parts) == 1:
                            task = f"[{file.removesuffix('.ts')}]"
                        elif len(path_parts) == 2:
                            folder_name = path_parts[0]
                            task = f"{folder_name}-[{file.removesuffix('.ts')}]"
                        else:
                            folder_name = path_parts[0]
                            sub_folder = path_parts[1]
                            task = f"{folder_name}-{{{sub_folder}}}-[{file.removesuffix('.ts')}]"
                            
                        add_file_directories(item, task, file_path)
                    else:
                        more += 1

print("More count:", more)

./docs/nest/integration
./docs/nest/sample
More count: 19


In [22]:
len(file_directories)
all_chunks = []
file_directories


[{'purpose': 'auto-mock',
  'task': '[bar.service]',
  'path': './docs/nest/integration\\auto-mock\\src\\bar.service.ts'},
 {'purpose': 'auto-mock',
  'task': '[foo.service]',
  'path': './docs/nest/integration\\auto-mock\\src\\foo.service.ts'},
 {'purpose': 'cors',
  'task': '[app.controller]',
  'path': './docs/nest/integration\\cors\\src\\app.controller.ts'},
 {'purpose': 'cors',
  'task': '[app.module]',
  'path': './docs/nest/integration\\cors\\src\\app.module.ts'},
 {'purpose': 'discovery',
  'task': '[app.module]',
  'path': './docs/nest/integration\\discovery\\src\\app.module.ts'},
 {'purpose': 'discovery',
  'task': '[webhooks.explorer]',
  'path': './docs/nest/integration\\discovery\\src\\webhooks.explorer.ts'},
 {'purpose': 'discovery',
  'task': 'decorators-[non-applied.decorator]',
  'path': './docs/nest/integration\\discovery\\src\\decorators\\non-applied.decorator.ts'},
 {'purpose': 'discovery',
  'task': 'decorators-[webhook.decorators]',
  'path': './docs/nest/integrat

In [23]:
for item in file_directories:
    file_path = item['path']
    purpose = item['purpose']
    task = item['task']
    
    if not os.path.exists(file_path):
        continue
        
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
        
    basename = os.path.basename(file_path)
    header = f"Framework: NestJs \n Purpose: {purpose}\t Task: {task}\n Content: \n"
    
    # Split by logical blocks (classes, functions, or double newlines) 
    # rather than arbitrary character/token cuts
    logical_blocks = content.split("\n\n")
    
    current_chunk_text = header
    
    for block in logical_blocks:
        # Check if adding this block exceeds typical block size approximation
        if len(tokenizer.encode(current_chunk_text + "\n\n" + block, add_special_tokens=False)) > BLOCK_SIZE:
            # Save the current chunk
            tokens = tokenizer.encode(current_chunk_text, add_special_tokens=False)
            chunk_text = tokenizer.decode(tokens, skip_special_tokens=True)
            
            all_chunks.append({
                "text": chunk_text,
                "framework": "Nest Js",
                "purpose": purpose,
                "task": task,
                "source": basename,
            })
            # Reset with header and current block
            current_chunk_text = header + block
        else:
            current_chunk_text += ("\n\n" + block if current_chunk_text != header else block)
            
    # Append any remaining text
    if current_chunk_text.strip() != header.strip():
        tokens = tokenizer.encode(current_chunk_text, add_special_tokens=False)
        chunk_text = tokenizer.decode(tokens, skip_special_tokens=True)
        all_chunks.append({
            "text": chunk_text,
            "framework": "Nest Js",
            "purpose": purpose,
            "task": task,
            "source": basename,
        })
        

In [24]:
output_file = "chunks/nestjs.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"Success! Generated {len(all_chunks)} chunks in {output_file}")

Success! Generated 664 chunks in chunks/nestjs.jsonl


In [25]:
# def collect_nest_files(base_dir, category):
#     """
#     Walks the given base directory (e.g. integration or sample)
#     and collects all relevant files and metadata.
#     """
#     collected = []
#     # Directories we want to skip to avoid noise/bloat in training data
#     skip_dirs = {"node_modules", "dist", "build", "coverage", ".git", ".circleci", ".github", "hooks"}
    
#     for root, dirs, files in os.walk(base_dir):
#         # In-place modify dirs to prune unwanted directories from traversal
#         dirs[:] = [d for d in dirs if d not in skip_dirs]
        
#         for file in files:
#             # Targets: typescript files, configurations, scripts, json metadata
#             if file.endswith((".ts", ".json", ".js", ".mjs")):
#                 # Avoid lockfiles
#                 if file in ("package-lock.json", "yarn.lock", "pnpm-lock.yaml"):
#                     continue
                    
#                 file_path = os.path.join(root, file)
#                 rel_path = os.path.relpath(file_path, nest_path).replace("\\", "/")
#                 parts = rel_path.split("/")
                
#                 # Project name is the folder right under integration/sample
#                 # E.g., 'integration/discovery/src/...' -> project name is 'discovery'
#                 project = parts[1] if len(parts) > 1 else ""
                
#                 collected.append({
#                     "absolute_path": file_path,
#                     "relative_path": rel_path,
#                     "category": category,
#                     "project": project,
#                     "filename": file
#                 })
                
#     return collected

# # Gather all files from integration and sample directories
# nest_files = []
# nest_files.extend(collect_nest_files(integration_path, "integration"))
# nest_files.extend(collect_nest_files(sample_path, "sample"))

# print(f"Total NestJS files found: {len(nest_files)}")
# # Let's inspect a few samples
# for item in nest_files[:5]:
#     print(f"- {item['relative_path']} | Project: {item['project']}")

In [ ]:
# chunks = []

# for file_info in nest_files:
#     try:
#         with open(file_info["absolute_path"], "r", encoding="utf-8") as f:
#             body = f.read().strip()
#     except Exception as e:
#         # Skip files that can't be read (e.g. binary or encoding issues)
#         continue
        
#     if not body:
#         continue
        
#     # Define purpose using the project name and filename
#     purpose = f"{file_info['project']} - {file_info['filename']}"
    
#     # Construct clean fine-tuning header containing only Framework, Category, and Purpose
#     header = (
#         f"Framework: NestJS \n"
#         f"Category: {file_info['category']} \n"
#         f"Purpose: {purpose} \n"
#         f"Content: \n"
#     )
    
#     full_text = header + body
    
#     # Tokenize the complete text
#     tokens = tokenizer.encode(full_text, add_special_tokens=False)
    
#     # Segment tokenized text using sliding window (BLOCK_SIZE + STRIDE)
#     for i in range(0, len(tokens), STRIDE):
#         chunk_tokens = tokens[i:i + BLOCK_SIZE]
#         chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
#         chunks.append({
#             "text": chunk_text,
#             "framework": "NestJS",
#             "source": file_info["relative_path"]
#         })

# output_file = "chunks/nestjs.jsonl"
# # Create chunks directory if it doesn't exist
# os.makedirs(os.path.dirname(output_file), exist_ok=True)

# with open(output_file, "w", encoding="utf-8") as f:
#     for chunk in chunks:
#         f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

# print(f"Success! Generated {len(chunks)} chunks in {output_file}")

NameError: name 'nest_files' is not defined